# Kilocalories analysis: recorded energy estimates and steps
For Friday October 23, 2026. This descriptive EDA reads only kilocalories, activity and steps. It does not change the database or other notebooks. Saved outputs contain aggregate statistics and participant-anonymous plots, not raw records or identifiers.

**Working interpretation, not a verified source definition:** `active_kilocalories` appears to be an activity-related energy estimate; `bmr_kilocalories` appears to be a basal-metabolism-related estimate. The names indicate kilocalories (kcal), but SQL types do not verify units, estimation algorithm, wear coverage or whether BMR is prorated. Do not add them and label the result an official total without source semantics. No medical or causal conclusions are intended.

## 1. Inspect the schema and establish a candidate row meaning
A synthetic row could mean “participant A, one labelled date and reporting period, active estimate 500 kcal, BMR estimate 1,500 kcal.” These are invented numbers. Actual grain must be tested: one participant/date or one participant/period? Calendar labels do not prove full-day recording.

`additional_id` is a candidate record identifier; user/group/type describe context. Calendar date/week/day label a record; start/end/duration describe its reporting window. We inspect every column without displaying real rows. Credentials stay in the private `.env`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, inspect, text
from dotenv import dotenv_values
from IPython.display import display
PROJECT=Path.cwd()
url=dotenv_values(PROJECT/'.env').get('DATABASE_URL')
assert url, 'Configure DATABASE_URL privately in .env.'
engine=create_engine(url); tables={}; schema=[]
try:
    inspector=inspect(engine)
    for name in ['kilocalories','activity','steps']:
        for col in inspector.get_columns(name):
            schema.append({'table':name,'column':col['name'],'sql_type':str(col['type'])})
        with engine.connect() as conn:
            tables[name]=pd.read_sql(text('SELECT * FROM '+name),conn)
finally:
    engine.dispose()
del url
for df in tables.values():
    df['calendar_date']=pd.to_datetime(df.calendar_date,errors='coerce')
k=tables['kilocalories'].copy()
metrics=['active_kilocalories','bmr_kilocalories']
display(pd.DataFrame(schema).query("table == 'kilocalories'"))
display(pd.DataFrame({'column':k.columns,'pandas_type':k.dtypes.astype(str).values,
 'null_rows':k.isna().sum().values,'distinct_nonnull':k.nunique().values}))
display(pd.DataFrame([{'table':name,'rows':len(df),'participants':df.sdv_user_id.nunique()} for name,df in tables.items()]))


## 2. Quality and candidate keys
Count missing, zero and negative energy values separately. Zero is recorded zero, not proof of no movement or no energy expenditure; negative values need a source/sentinel definition. Do not replace nulls with zero. Test keys in each table before joining; equal row counts alone cannot establish correspondence.

In [ ]:
period_key=['sdv_user_id','calendar_date','start_measure_period_sec','end_measure_period_sec']
def valid_key(df,key):
    return not df[key].isna().any(axis=None) and not df.duplicated(key).any()
audit=[]
for name,df in tables.items():
    for label,key in [('record_id',['additional_id']),('participant_date',['sdv_user_id','calendar_date']),
                      ('participant_period',['sdv_user_id','start_measure_period_sec','end_measure_period_sec']),
                      ('participant_date_period',period_key)]:
        audit.append({'table':name,'key':label,'null_key_rows':int(df[key].isna().any(axis=1).sum()),
          'rows_in_duplicate_keys':int(df.duplicated(key,keep=False).sum()),'valid_unique_key':valid_key(df,key)})
display(pd.DataFrame(audit))
print('Exact duplicate kilocalories rows after first:',int(k.duplicated().sum()))
display(pd.DataFrame([{'metric':m,'null':int(k[m].isna().sum()),'zero':int(k[m].eq(0).sum()),
 'negative':int(k[m].lt(0).sum()),'positive':int(k[m].gt(0).sum())} for m in metrics]))
display(k[metrics].describe().T)


## 3. Measurement periods, overlap and observed-date coverage
Compare duration with end minus start, flag invalid periods and inspect the distribution of hours. The Unix-seconds conversion is a plausibility diagnostic; timezone and date-label conventions still require confirmation. A 24-hour window does not measure device wear.

An overlap means a start is earlier than the latest preceding end for the same participant. Endpoint equality is not counted as overlap here; inclusive/exclusive conventions remain unknown. Coverage is observed unique dates divided by dates between each participant's first and last observation; it excludes unknown enrolment boundaries and does not turn missing dates into zeros.

In [ ]:
k['elapsed_sec']=k.end_measure_period_sec-k.start_measure_period_sec
k['period_hours']=k.duration_measure_period_sec/3600
ordered=k.sort_values(period_key)
previous_end=ordered.groupby('sdv_user_id').end_measure_period_sec.transform(lambda x:x.cummax().shift())
utc_start=pd.to_datetime(k.start_measure_period_sec,unit='s',utc=True,errors='coerce')
display(pd.DataFrame([{'nonpositive_duration':int(k.duration_measure_period_sec.le(0).sum()),
 'end_not_after_start':int(k.elapsed_sec.le(0).sum()),
 'duration_equals_end_minus_start':int(k.duration_measure_period_sec.eq(k.elapsed_sec).sum()),
 'duration_equals_end_minus_start_plus_one':int(k.duration_measure_period_sec.eq(k.elapsed_sec+1).sum()),
 'overlapping_period_rows':int(ordered.start_measure_period_sec.lt(previous_end).sum()),
 'unparseable_unix_second_starts':int(utc_start.isna().sum()),
 'UTC_start_date_matches_label':int(utc_start.dt.tz_localize(None).dt.normalize().eq(k.calendar_date).sum())}]))
display(k.period_hours.value_counts().sort_index().rename_axis('reported_hours').to_frame('records'))
coverage=k.groupby('sdv_user_id').agg(first=('calendar_date','min'),last=('calendar_date','max'),
 observed_dates=('calendar_date','nunique'))
coverage['span_days']=(coverage['last']-coverage['first']).dt.days+1
coverage['missing_dates_within_span']=coverage.span_days-coverage.observed_dates
coverage['observed_date_percent']=100*coverage.observed_dates/coverage.span_days
display(coverage[['observed_dates','span_days','missing_dates_within_span','observed_date_percent']].describe().T)


## 4. Energy estimate distributions and unequal follow-up
Histograms show record-weighted distributions: more-recorded participants contribute more. We also summarize participant medians without displaying identifiers. BMR stability or variation cannot establish how its estimate was calculated. Do not interpret short-period values as comparable daily totals or annualize them.

In [ ]:
display(k.groupby('sdv_user_id')[metrics].median().describe().T.rename_axis('distribution_of_participant_medians'))
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,m in zip(axes,metrics):
    k[m].dropna().plot.hist(bins=25,ax=ax)
    ax.set_title(m); ax.set_xlabel('Recorded kcal (source definition unconfirmed)'); ax.set_ylabel('Records')
plt.tight_layout(); plt.show()
display(k.assign(period_scope=np.where(k.period_hours.eq(24),'exactly 24 hours','other duration')).groupby('period_scope')[metrics].agg(['count','median','min','max']))


## 5. Validate relationships with activity and steps
Outer join on participant/date/start/end, require unique non-null keys on both sides, and enforce one-to-one cardinality. Count unmatched periods and compare record ID, group and duration. Skip ambiguous joins instead of deduplicating. The activity and steps metrics are compared on their own exact-period join as an independent consistency check.

In [ ]:
joined_tables={}; joins=[]; agreement=[]
for name in ['activity','steps']:
    right=tables[name]
    if not (valid_key(k,period_key) and valid_key(right,period_key)):
        print(name+': join skipped because keys fail uniqueness/null checks.'); continue
    j=k.merge(right,on=period_key,how='outer',suffixes=('_kcal','_other'),indicator=True,validate='one_to_one')
    joined_tables[name]=j
    both=j._merge.eq('both')
    joins.append({'other_table':name,'matched':int(both.sum()),'kcal_only':int(j._merge.eq('left_only').sum()),
                 'other_only':int(j._merge.eq('right_only').sum())})
    for field in ['additional_id','sdv_group_id','duration_measure_period_sec']:
        a=j.loc[both,field+'_kcal']; b=j.loc[both,field+'_other']; present=a.notna() & b.notna()
        agreement.append({'table':name,'field':field,'both_nonnull':int(present.sum()),
          'equal':int((present & a.eq(b)).sum()),'different':int((present & a.ne(b)).sum()),'missing_either':int((~present).sum())})
display(pd.DataFrame(joins)); display(pd.DataFrame(agreement))
if valid_key(tables['activity'],period_key) and valid_key(tables['steps'],period_key):
    cross=tables['activity'][period_key+['steps']].merge(tables['steps'][period_key+['steps']],
        on=period_key,how='outer',suffixes=('_activity','_steps'),indicator=True,validate='one_to_one')
    both=cross._merge.eq('both'); present=cross.steps_activity.notna() & cross.steps_steps.notna()
    display(pd.DataFrame([{'activity_steps_matched':int(both.sum()),'unmatched':int((~both).sum()),
      'equal_nonnull_steps':int((both & present & cross.steps_activity.eq(cross.steps_steps)).sum()),
      'different_nonnull_steps':int((both & present & cross.steps_activity.ne(cross.steps_steps)).sum()),
      'matched_missing_steps':int((both & ~present).sum())}]))


## 6. Active kcal versus steps: pooled and within-participant views
Use matched, context-agreeing periods with nonmissing, nonnegative steps and active kcal. Recorded zeros remain eligible. Show all eligible periods, then exactly-24-hour and strictly-positive 24-hour sensitivity subsets. Excluding zeros changes the question and is reported, not hidden.

Pearson describes linear association; Spearman describes rank association. Pooled correlation can reflect differences between participants, period lengths and unequal follow-up. Within-participant centering subtracts each participant's own subset mean before correlation; it removes mean-level differences but still weights participants by observed records and does not remove time trends or wear bias. We also summarize participant-specific correlations where variation and at least three observations permit them. No p-values, causal claim or predictive validation is implied. Scatter markers are anonymous records, with no IDs or dates.

In [ ]:
correlation_rows=[]; per_participant_rows=[]
if 'steps' in joined_tables:
    j=joined_tables['steps']
    context=j.sdv_group_id_kcal.eq(j.sdv_group_id_other) & j.duration_measure_period_sec_kcal.eq(j.duration_measure_period_sec_other)
    matched=j._merge.eq('both') & context
    eligible=matched & j.steps.notna() & j.steps.ge(0) & j.active_kilocalories.notna() & j.active_kilocalories.ge(0)
    display(pd.DataFrame([{'matched_context_agrees':int(matched.sum()),
      'excluded_missing_or_negative_metrics':int((matched & ~eligible).sum()),
      'eligible_zero_steps':int((eligible & j.steps.eq(0)).sum()),
      'eligible_zero_active_kcal':int((eligible & j.active_kilocalories.eq(0)).sum())}]))
    def corr(frame,x,y,method='pearson'):
        return frame[x].corr(frame[y],method=method) if len(frame)>=3 and frame[x].nunique()>1 and frame[y].nunique()>1 else np.nan
    scopes={'all eligible periods':eligible,'exactly 24 hours':eligible & j.duration_measure_period_sec_kcal.eq(86400),
        '24 hours, both metrics positive':eligible & j.duration_measure_period_sec_kcal.eq(86400) & j.steps.gt(0) & j.active_kilocalories.gt(0)}
    for label,mask in scopes.items():
        d=j.loc[mask].copy()
        for metric in ['steps','active_kilocalories']:
            d[metric+'_centered']=d[metric]-d.groupby('sdv_user_id')[metric].transform('mean')
        correlation_rows.append({'scope':label,'records':len(d),'participants':d.sdv_user_id.nunique(),
          'pooled_Pearson':corr(d,'steps','active_kilocalories'),
          'pooled_Spearman':corr(d,'steps','active_kilocalories','spearman'),
          'within_centered_Pearson':corr(d,'steps_centered','active_kilocalories_centered')})
        for _,part in d.groupby('sdv_user_id'):
            per_participant_rows.append({'scope':label,'records':len(part),'Pearson':corr(part,'steps','active_kilocalories')})
    display(pd.DataFrame(correlation_rows))
    pp=pd.DataFrame(per_participant_rows)
    display(pp.groupby('scope').Pearson.agg(['count','median','min','max']).rename(columns={'count':'participants_with_defined_correlation'}))
    d=j.loc[scopes['exactly 24 hours']].copy()
    for metric in ['steps','active_kilocalories']:
        d[metric+'_centered']=d[metric]-d.groupby('sdv_user_id')[metric].transform('mean')
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    axes[0].scatter(d.steps,d.active_kilocalories,alpha=.35,s=15)
    axes[0].set(xlabel='Recorded steps',ylabel='Recorded active kcal',title='24-hour windows: pooled records')
    axes[1].scatter(d.steps_centered,d.active_kilocalories_centered,alpha=.35,s=15)
    axes[1].axhline(0,color='gray',lw=.7); axes[1].axvline(0,color='gray',lw=.7)
    axes[1].set(xlabel='Steps minus participant mean',ylabel='Active kcal minus participant mean',title='24-hour windows: within-person deviations')
    plt.tight_layout(); plt.show()
else:
    print('Correlation skipped: steps join not validated.')


## Verified findings from this run
- **987 records across 8 participants**, with no missing kilocalories-table values, exact duplicate rows or null/duplicate tested keys. Active kcal contains **21 zeros**, no negatives; BMR is positive in all records.
- Recorded active kcal: median **316**, range **0–2,381**. Recorded BMR: median **1,795**, range **4–2,414**. These are window-level descriptive estimates, not verified daily expenditure totals.
- **970** windows report exactly 24 hours; **17** have other durations. All durations equal end minus start, and **2** windows overlap a preceding period. No UTC start date matches its calendar label under the Unix-seconds interpretation; confirm timezone conventions before treating this as an error.
- Observed dates per participant range **16–250**. Within first-to-last observed-date spans, date coverage ranges **89.36%–100%**; neither measure establishes wear time or enrolment coverage.
- Exact-period joins match **987/987** records to activity and steps with no unmatched keys. Compared record identifiers, groups and durations agree. Activity steps and steps-table steps also agree for all 987 records.
- All **987 matched records** have nonmissing, nonnegative steps and active kcal. Pooled Pearson is **0.610** and within-participant-centered Pearson is **0.671**. For **970 exactly-24-hour windows**, the corresponding values are **0.607** and **0.668** (pooled Spearman **0.586**). Restricting those windows to both metrics positive leaves **952 records**, with Pearson **0.595** pooled and **0.654** centered.
- Participant-specific Pearson correlations for 24-hour windows range **0.478–0.948** (median **0.695**, eight defined correlations). This heterogeneity and unequal follow-up matter; none of these correlations is causal or a model-validation result.

**Takeaway:** More recorded steps tend to accompany higher active-kcal estimates, including within participants, but the magnitude varies and the estimates may share movement inputs. Period quality, wear coverage and undocumented energy semantics limit interpretation.


## 7. Teacher presentation and limits
“I checked the record grain and period quality, then used validated one-to-one period joins before comparing active calorie estimates with steps. I separated pooled association from within-participant deviations and checked whether zeros or non-24-hour windows changed the result.”

Report correlation together with the subset denominator and participant count. This is a descriptive relationship between recorded estimates. Device algorithms may themselves use movement information, so agreement is not independent validation. Missing dates, overlap, partial windows, repeated observations and unequal coverage limit interpretation. No medical inference follows. `active + bmr` is not labelled an official total.

Ask the owner to confirm kcal units, active/BMR definitions and estimation methods, BMR prorating, timezone/endpoints, calendar labels, actual wear time and expected enrolment coverage. Goals intensity semantics also remain unverified in the separate goals review. CSV, RAG and modelling remain deferred.